# Lesson 5: From notebooks to a product: the FieldSense package, evidence labels and the field barcode

In Lessons 1–4 you produced real results: corrected rainfall, soil with uncertainty, soil moisture, and ten years of greenness for the four Elora plots. But the code that made them was spread across four notebooks. A website can't run a notebook.

In this lesson we turn that work into **something the product can use**:

| What we build | What it is | Where the demo uses it |
|---|---|---|
| **The `fieldsense` package** | All the code from Lessons 1–4, cleaned up and stored in your GitHub repository | The backend API (Lesson 10) imports it |
| **Automated tests** | 19 small checks that run in seconds | Proves to you, your team and the judges that the code does what it says |
| **The evidence registry** | Every data source with its label: *Measured / Validated / Modelled / Assumed* | The "evidence passport" in the app |
| **The field barcode** | One picture showing 11 years of a field's life | The opening image of the demo |
| **Field profiles** | One JSON file per plot with everything we know, each number with a range and its evidence | What the website reads |

**New ideas in this lesson**
- *ML operations:* packages, version control, **tests**, and **data contracts** (a fixed format that two parts of a system agree on)
- *Product design:* show a whole history in one glance; label how sure you are, number by number
- *Science:* living soil cover through the year as a soil-health signal

**Time:** about 20 minutes. No big downloads: this lesson reads files that Lessons 1–4 saved to your Drive.


## Project setup: run this first, every session

Everything we save, **data files and code modules**, goes into your **Google Drive** under `MyDrive/fieldsense/`. Colab wipes its own disk whenever a session ends, but Drive keeps everything, so each lesson can pick up the previous lessons' files.

Colab will ask permission to access your Drive the first time each session; click **Allow**.


In [ ]:
# ---- Project setup: run this first, every session ----
import os, sys, glob, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/fieldsense"
except ImportError:                       # running outside Colab: use a local folder instead
    PROJECT_DIR = os.path.abspath("fieldsense")
CACHE_DIR = os.path.join(PROJECT_DIR, "data_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)                     # %%writefile now saves code modules into Drive too
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)       # lets `import fieldsense_data` find them

# One-time rescue: copy anything still sitting in Colab's temporary folder into Drive
for f in glob.glob("/content/data_cache/*"):
    if not os.path.exists(os.path.join(CACHE_DIR, os.path.basename(f))):
        shutil.copy(f, CACHE_DIR)
        print("Rescued into Drive:", os.path.basename(f))

print("Project folder:", PROJECT_DIR)
print("Saved data files:", sorted(os.listdir(CACHE_DIR)) or "none yet")
PRIVATE_DIR = os.path.join(PROJECT_DIR, "private")       # lab records: NEVER goes to GitHub
PROFILE_DIR = os.path.join(PROJECT_DIR, "profiles")      # field profiles + evidence registry
DEMO_DIR = os.path.join(PROJECT_DIR, "demo")             # figures for slides and website
for d in (PRIVATE_DIR, PROFILE_DIR, DEMO_DIR):
    os.makedirs(d, exist_ok=True)

## Step 1: Why a package? (read this before running anything)

Until now, each lesson used `%%writefile` to save a code file like `fieldsense_data.py` into Drive, and later notebooks imported it. That worked for learning, but it has three problems for a product:

1. **No history.** If a change breaks something, you can't see what changed or go back.
2. **No sharing.** Nikolett and Daniella would need your Drive to run anything.
3. **No checks.** Nothing tells you when a change breaks an older result.

A **package** fixes all three. It's a folder of code with a standard layout, so Python (and anyone) can install it with one command. It lives in **GitHub**, which records every change (*version control*). And it comes with **tests**.

Here is how the repository is organised. I wrote it from your Lesson 1–4 code; nothing new was invented, only cleaned up:

```
FieldSense/
├── pyproject.toml        the package's "ID card": name, version, which libraries it needs
├── README.md             the front page on GitHub: what it is, how to install, data sources
├── LICENSE               Apache 2.0 (Space Apps asks for open source)
├── .gitignore            files git must NEVER upload: private/, data_cache/, credentials
├── src/fieldsense/
│   ├── sources/          one file per data source, each downloads + caches
│   │   ├── power.py        NASA POWER weather          (Lesson 1)
│   │   ├── stations.py     Environment Canada stations (Lesson 2)
│   │   ├── soilgrids.py    SoilGrids soil              (Lesson 2)
│   │   ├── appeears.py     AppEEARS / SMAP             (Lesson 3)
│   │   └── hls.py          HLS satellite greenness     (Lesson 4)
│   ├── correction.py     rainfall bias correction + held-out test (Lesson 2)
│   ├── phenology.py      crop history, barcode, living cover      (Lesson 4 + NEW)
│   ├── evidence.py       evidence labels                          (NEW)
│   ├── profile.py        field profile JSON                       (NEW)
│   └── viz.py            figures                                  (NEW)
├── tests/test_core.py    the automated checks
└── notebooks/            this notebook
```

**Two separate homes, on purpose:**
- **Code → GitHub.** Small, text, public, versioned.
- **Data → Google Drive.** Big, partly private, re-downloadable. The `.gitignore` file makes sure data never leaks into GitHub by accident.

**Why `src/fieldsense/` and not just `fieldsense/`?** With the extra `src` folder, Python can only find the package after it's properly *installed*. That sounds annoying, but it means the tests check the installed package, exactly as the website will use it, and not some stray file that happens to sit in the current folder.


## Step 2: Get the code from GitHub and install it

This cell does three things:

1. **`git clone`** downloads the repository to Colab's own disk (`/content/FieldSense`). If it's already there from earlier in this session, **`git pull`** fetches only what changed.
   - Why not clone into Drive? Git creates thousands of tiny files, and Drive is very slow with those. The code is safe in GitHub anyway, so a temporary copy is fine.
2. **`pip install -e ".[dev]"`** installs the package.
   - `-e` means *editable*: Python uses the files in the folder directly, so if you edit a `.py` file, the change applies without reinstalling.
   - `[dev]` is an *optional extra*: it also installs `pytest` (the test runner). The satellite libraries are another extra, `[hls]`; this lesson doesn't need them because it reads the greenness numbers Lesson 4 already saved.
3. Adds `src/` to Python's search path. Normally the install does this, but a notebook that is *already running* doesn't notice new installs until it restarts. This line saves you a restart.

**If your repository is private:** add a Colab Secret called `GITHUB_TOKEN` (a GitHub *personal access token* with read access, made at github.com → Settings → Developer settings). The cell uses it automatically. Never paste a token into the code.


In [ ]:
import subprocess
REPO_URL = "https://github.com/SetarehAlamdar/FieldSense.git"
REPO_DIR = os.environ.get("FIELDSENSE_REPO", "/content/FieldSense")

def sh(cmd):
    """Run a shell command; print its output; stop the notebook if it fails."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

url = REPO_URL
try:                                           # private repo: use a token from Colab Secrets
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                                       # public repo (or not in Colab): no token needed

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"git clone --quiet {url} {REPO_DIR}")
sh(f'pip install -q -e "{REPO_DIR}[dev]"')

SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import fieldsense as fs
print("fieldsense version", fs.__version__, "from", os.path.dirname(fs.__file__))
sh(f"git -C {REPO_DIR} log --oneline -5")

**What you should see:** the version (`0.1.0`), the folder it was loaded from, and the latest *commits*.

A **commit** is a saved snapshot of the code with a message saying what changed. That short code at the start of each line (like `a1b2c3d`) is the commit's ID. When you report a result ("barcode made with commit a1b2c3d"), anyone can get exactly that code back. That's what **reproducible** means in ML operations.


## Step 3: Run the tests

A **test** is a tiny program that feeds the code an input where *we already know the right answer*, then checks the code gets it. For example:

- *"If a model is always exactly 20% too low, the held-out error must be −20% before correction and 0% after."*
- *"Sand 90%, silt 5%, clay 5% must be classed as sand."*
- *"A fake season that peaks on day 170 must be labelled winter wheat; one that peaks on day 215 must not."*
- *"Across an 82-day cloud gap, daily greenness must stay missing, never invented."*
- *"A number marked 'validated' must say how it was tested, or the code refuses it."*

`pytest` finds every function whose name starts with `test_` and runs it. A dot `.` means passed, `F` means failed.

**Why this matters for the hackathon:** on 14–15 November, five people will edit this code at the same time under pressure. If someone's change breaks the wheat rule, the tests will tell you in 1 second, instead of you finding out during the live demo.

**The limit to remember from Lesson 4:** a test only checks the code against *what we believe*. The wheat threshold of 0.5 passed my synthetic tests and was still wrong on real data, because the test was built on the same wrong belief. So tests catch **code bugs**; only real ground truth catches **wrong assumptions**. We need both.


In [ ]:
sh(f"cd {REPO_DIR} && python -m pytest -q")

**What you should see:** `19 passed`. If anything fails, stop here and tell me the message; nothing below should be trusted until the tests pass.

Want to see what a test looks like? Open the **Files** panel on the left → `/content/FieldSense/tests/test_core.py`. Each test is 3–8 lines, and the name says what it checks.


## Step 4: Load what Lessons 1–4 saved

The package contains the *methods*. The *results* are in your Drive. This cell looks for each file and tells you which lesson makes it if it's missing.

| File | Made in | What's in it |
|---|---|---|
| `weather_elora_corrected.csv` | Lesson 2, Step 10 | Daily NASA POWER weather, with corrected rain in `PRECIP_CORR` |
| `fieldsense_config.json` | Lesson 2, Step 10 | The correction's factors and its held-out test result |
| `soilgrids_*.csv` | Lesson 2, Part B | Soil properties with 5%–95% ranges |
| `smap_l4_elora_daily.csv` | Lesson 3 | Daily soil moisture (optional) |
| `hls_plot_ndvi.csv` | Lesson 4 | Every clear satellite look at every plot |


In [ ]:
import json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find(name, lesson, required=True):
    hits = sorted(glob.glob(os.path.join(CACHE_DIR, name)))
    status = "found" if hits else ("MISSING (needed)" if required else "missing (optional)")
    print(f"{name:32s} {status:20s} <- {lesson}")
    return hits[0] if hits else None

f_weather = find("weather_elora_corrected.csv", "Lesson 2, Step 10")
f_config  = find("fieldsense_config.json",     "Lesson 2, Step 10")
f_soil    = find("soilgrids_*.csv",            "Lesson 2, Part B")
f_smap    = find("smap_l4_elora_daily.csv",    "Lesson 3", required=False)
f_hls     = find("hls_plot_ndvi.csv",          "Lesson 4")

weather = pd.read_csv(f_weather, index_col=0, parse_dates=True)
config = json.load(open(f_config))
soil_table = pd.read_csv(f_soil)
smap = pd.read_csv(f_smap, index_col=0, parse_dates=True) if f_smap else None
hls = pd.read_csv(f_hls, parse_dates=["date"])
hls = hls[hls["ndvi"].notna()]

LAT, LON = config["location"]["lat"], config["location"]["lon"]
PLOTS = sorted(int(p) for p in hls["plot"].unique())
print(f"\nWeather {weather.index.min():%Y}-{weather.index.max():%Y}; "
      f"{len(hls)} clear plot-looks for plots {PLOTS}, {hls.date.min():%Y}-{hls.date.max():%Y}")

## Step 5: The evidence registry, deciding how much to trust each source

This is the heart of what makes FieldSense different. Most teams will show numbers. We show **numbers plus how we know them**.

Every data source gets one of four labels, from strongest to weakest:

| Label | Rule | |
|---|---|---|
| **Measured** | Observed directly at this field | strongest |
| **Validated** | Modelled, **and** checked against independent ground truth, with the result written down | |
| **Modelled** | From a model, not checked here | |
| **Assumed** | From literature or expert rules, not tested here | weakest |

The code **enforces** the rules: you can't create a "validated" item without saying how it was tested and what the result was (one of the tests checks exactly this). That stops anyone, including us at 3 a.m. during the hackathon, from labelling something "validated" just because it sounds better.

**Let's decide each label together:**

- **Rainfall (NASA POWER, corrected) → Validated.** In Lesson 2 we fitted the correction on 1996–2010 and tested on 2011–2025, years it never saw, against the Fergus station. The result is read from your config file, not typed by hand.
- **Temperature (NASA POWER) → Modelled.** Be honest: we never checked temperature against a station. It's probably fine (temperature varies smoothly over 50 km, rain doesn't), but "probably fine" is not "validated". Checking it is a 10-minute job for later, and then the label can be upgraded.
- **Soil (SoilGrids) → Modelled.** It's a machine-learning prediction at 250 m. Remember the very wide organic-carbon range at Elora.
- **Soil moisture (SMAP L4) → Modelled.** A model at 9 km, combining satellite readings with weather.
- **Greenness (HLS) → Measured.** Real pixels, from inside each plot, with mixed edge pixels removed.
- **Crop history from greenness → Validated *if* your private lab file is present, otherwise Modelled.** The cell computes the check privately and stores **only the aggregate score** ("wheat detected in X of Y"), never the plot-by-plot records.


In [ ]:
from fieldsense import Evidence, EvidenceRegistry
from fieldsense.phenology import field_history

reg = EvidenceRegistry()
pc = config["precip_correction"]

reg.add(Evidence(
    "power_precip_corrected", "validated",
    claim="Daily rainfall at the field, from NASA POWER, corrected month by month to match local stations.",
    source="NASA POWER (PRECTOTCORR), corrected against ECCC Fergus Shand Dam (4760)",
    test="Correction fitted on 1996-2010, tested on 2011-2025 (years it never saw)",
    result=f"Held-out error after correction: {pc['held_out_test_2011_2025_error_pct']:+.1f}% of station rainfall",
    caveat=f"~50 km grid: single storms are missed; good for seasons, not days. {pc.get('cross_check', '')}"))

reg.add(Evidence(
    "power_temperature", "modelled",
    claim="Daily air temperature at the field (frost dates, heat units).",
    source="NASA POWER (T2M, T2M_MIN, T2M_MAX)",
    caveat="Not yet checked against a station. ~50 km grid; low-lying fields may be colder on still nights."))

reg.add(Evidence(
    "soilgrids_topsoil", "modelled",
    claim="Topsoil (0-30 cm) texture, organic carbon, pH and bulk density, each with a 90% range.",
    source="ISRIC SoilGrids 2.0, 250 m (Poggio et al., 2021)",
    caveat="A prediction, not a sample. The organic-carbon range is very wide here; a lab soil test should replace it."))

if smap is not None:
    reg.add(Evidence(
        "smap_l4_rootzone", "modelled",
        claim="Typical root-zone soil wetness by month.",
        source="NASA SMAP L4 (SPL4SMGP v8) via AppEEARS, 9 km",
        caveat="One value for a 9 km square: shows wet and dry YEARS well, not wet and dry spots in a field."))

reg.add(Evidence(
    "hls_greenness", "measured",
    claim="Greenness (NDVI) of each field on every clear satellite pass since 2015.",
    source="NASA HLS v2.0, Landsat 8/9 (L30) + Sentinel-2 (S30), 30 m",
    caveat="Clouds and snow leave gaps; gaps longer than 30 days are left empty, never filled in."))

# Crop history: validated against private records if available (AGGREGATE ONLY)
history = {p: field_history(hls[hls["plot"] == p][["date", "ndvi"]]).assign(plot=p) for p in PLOTS}
all_hist = pd.concat(history.values())
gt_file = os.path.join(PRIVATE_DIR, "elora_ground_truth_PRIVATE.json")
if os.path.exists(gt_file):
    gt = pd.DataFrame(json.load(open(gt_file))["plot_years"])
    v = gt.merge(all_hist, on=["plot", "year"])
    v = v[v.satellite_says != "too few clear images"]
    truth_w, sat_w = v.crop.eq("winter wheat"), v.satellite_says.str.startswith("winter wheat")
    sat_cs = v.satellite_says.str.startswith("corn or soybean")
    score = (f"winter wheat found in {int((truth_w & sat_w).sum())} of {int(truth_w.sum())} wheat plot-years; "
             f"corn/soybean labelled correctly in {int((~truth_w & sat_cs).sum())} of {int((~truth_w).sum())}")
    del gt, v                                   # drop the private table from memory straight away
    reg.add(Evidence(
        "hls_crop_history", "validated",
        claim="Which crop grew each year (winter wheat vs corn/soybean) and whether there was green cover off-season.",
        source="Derived from NASA HLS greenness timing (fieldsense.phenology)",
        test="Compared with partner field records, Elora rotation plots, 2015-2021",
        result=score,
        caveat="Thresholds were tuned at this site, and wheat cases are few; corn and soybean are not separated."))
else:
    reg.add(Evidence(
        "hls_crop_history", "modelled",
        claim="Which crop grew each year (winter wheat vs corn/soybean) and whether there was green cover off-season.",
        source="Derived from NASA HLS greenness timing (fieldsense.phenology)",
        caveat="No ground truth available in this run; rules come from Elora and may not transfer."))

reg.to_json(os.path.join(PROFILE_DIR, "evidence.json"))
pd.set_option("display.max_colwidth", 90)
display(reg.table()[["id", "label", "result", "caveat"]])
print("Saved:", os.path.join(PROFILE_DIR, "evidence.json"))

**Read the table out loud as if you were a judge.** Every row answers *"how do you know?"* and *"what's the catch?"*. That's what the challenge means by *evidence* and *information gaps*, and the app will be able to show it automatically.

Notice one more design idea in the package: `reg.weakest([...])`. If a recommendation uses rainfall (validated) **and** soil (modelled), it can only be as trustworthy as **modelled**. A chain is only as strong as its weakest link. The rotation engine (Lesson 8) will use this to label every recommendation.

**Privacy check:** the lab records were read, compared, and deleted from memory in the same cell. What was saved is one sentence of counts. That's safe to show on a slide.


## Step 6: The field barcode

**The product idea:** a farmer, or a judge, should understand a field's whole history in **five seconds**, without reading a table.

So we draw each year as a horizontal stripe, January on the left to December on the right. Each day is coloured by greenness: **brown = bare soil, green = growing plants, grey = no clear look** (clouds, snow). Stack the years and you get something that looks like a barcode, and every field has its own.

**How the numbers become a picture (three functions in `phenology.py`):**
1. **`daily_ndvi`**: satellites see the field every few days, not every day. We draw a straight line between two clear looks to fill the days in between. **But** if two looks are more than 30 days apart (a cloudy spell, winter snow), we leave the gap **empty**. Inventing 6 weeks of greenness would be lying with a pretty picture.
2. **`barcode_matrix`**: arranges the days into a table, one row per year, 365 columns. (29 February is dropped so every row has the same length.)
3. **`viz.plot_field_barcode`**: draws the table. The label on the right is what the satellite thinks grew that year.

**How to read it:**
- A **wide green block in June** with brown after it → winter wheat (harvested in July).
- A **green block from July to September** → corn or soybean.
- **Green in October–November or in April** → a cover crop, or wheat sown in the fall. Soil covered when it would otherwise be bare.


In [ ]:
from fieldsense import viz
from fieldsense.phenology import daily_ndvi, barcode_matrix

ROTATION = {1: "simple", 2: "simple", 3: "diverse", 4: "diverse"}   # design since 2018 (Lesson 4)
short = lambda s: (s.replace("corn or soybean", "corn/soy").replace("winter wheat", "wheat")
                    .replace(" (overwintered cover crop?)", "").replace(" (cover crop?)", "")
                    .replace("green in spring, green after harvest", "spring+fall cover")
                    .replace("green in spring", "spring cover").replace("green after harvest", "fall cover"))

matrices = {}
fig, axes = plt.subplots(len(PLOTS), 1, figsize=(12, 3.3 * len(PLOTS)), layout="constrained")
for ax, p in zip(np.atleast_1d(axes), PLOTS):
    matrices[p] = barcode_matrix(daily_ndvi(hls[hls["plot"] == p][["date", "ndvi"]]))
    labels = dict(zip(history[p].year, history[p].satellite_says.map(short)))
    viz.plot_field_barcode(matrices[p], title=f"Plot {p}: {ROTATION.get(p, '?')} rotation", labels=labels, ax=ax)
sm = plt.cm.ScalarMappable(cmap=viz.BARCODE_CMAP, norm=plt.Normalize(0.1, 0.9))
fig.colorbar(sm, ax=axes, orientation="horizontal", location="bottom", fraction=0.02, aspect=60,
             label="greenness (NDVI):  brown = bare soil,  green = growing plants,  grey = no clear look")
fig.savefig(os.path.join(DEMO_DIR, "field_barcodes_elora.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(DEMO_DIR, "field_barcodes_elora.svg"), bbox_inches="tight")
plt.show()
print("Saved PNG (slides) and SVG (website) to", DEMO_DIR)

**Look at it before moving on. Two questions for you:**
1. Since 2018, can you see the difference between the simple rotation plots (1–2) and the diverse ones (3–4) *just from the colours*? Where is it?
2. Find a year with lots of grey. What does that do to our confidence for that year?

We save two formats on purpose: **PNG** is a fixed picture for slides. **SVG** is a drawing made of shapes, so it stays sharp at any size and the website can style it.

**Why this is different from other teams:** most teams will show a map with a colour layer, or an NDVI line chart. A line chart of 11 years is unreadable; a barcode isn't. And it's built only from **measured** data, so it's the strongest evidence we have.


## Step 7: Living cover, turning the barcode into numbers

A picture convinces people; numbers let us compare rotations. Soil scientists care about **how many days the soil has living roots**, especially *outside* the main crop season, because that's when bare soil erodes and loses nitrogen. This is a key reason farmers grow cover crops and wheat in rotation.

`cover_summary` counts, per year, from 1 March to 30 November (winter is mostly snow):

- `observed_days`: days with a value (not inside a long cloud gap)
- `bare_days`: greenness below 0.30 (bare soil or crop residue)
- `green_days`: greenness 0.40 or higher (clearly living plants)
- `green_days_off_season`: living cover **outside** 15 May – 30 Sep, the soil-health signal
- `green_share_off_season`: the same, as a **share of the off-season days we actually saw**

**Why a share and not a count?** A cloudy year has fewer observed days, so its raw counts are lower even if nothing changed in the field. Dividing by what we saw makes years and fields comparable. This is the same idea as Lesson 2's rule of dropping months with missing days.


In [ ]:
from fieldsense.phenology import cover_summary

covers = {p: cover_summary(m) for p, m in matrices.items()}
cov = pd.concat({p: c for p, c in covers.items()}, names=["plot", "year"]).reset_index()
cov["rotation"] = cov["plot"].map(ROTATION)
recent = cov[cov.year >= 2018]

summary = recent.groupby("rotation")[["bare_days", "green_days", "green_share_off_season"]].mean().round(2)
summary["green_share_off_season"] = (100 * summary["green_share_off_season"]).round(0).astype(int).astype(str) + "%"
print("Average per year since 2018 (1 Mar - 30 Nov):")
display(summary)

pivot = recent.pivot(index="year", columns="plot", values="green_share_off_season")
ax = (100 * pivot).plot(marker="o", figsize=(8, 3.5))
ax.set_ylabel("% of off-season days green"); ax.set_title("Living cover outside the main crop season", loc="left")
ax.legend([f"Plot {p} ({ROTATION.get(p)})" for p in pivot.columns], frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(DEMO_DIR, "living_cover_elora.png"), dpi=200); plt.show()

**What to look for:** if the diverse rotation (wheat + cover crops) really keeps the soil covered longer, its off-season green share should be clearly higher than the simple rotation's. If it is, that's a result built entirely from NASA data that agrees with what the lab designed, which is a strong demo moment.

**The honest caveat**, which goes into the evidence: four plots at one site is a small sample. It shows the method can see the difference; it doesn't prove the size of the effect elsewhere.


## Step 8: Climate, soil and soil moisture, every number with a range

Now we summarise the other sources with **one rule**: no bare numbers. Each number is stored as

```python
{"mid": 912.4, "low": 781.0, "high": 1060.2, "unit": "mm",
 "range_meaning": "10th-90th percentile across years", "evidence": "power_precip_corrected"}
```

`range_meaning` matters because "the range" means different things for different sources:
- For **climate**, low/high is **natural year-to-year variation**: in 1 year out of 10, rain is below `low`. That's not an error; it's the weather.
- For **soil**, low/high is **uncertainty**: SoilGrids isn't sure, and the true value is probably in between.

Mixing these up is a classic mistake. A farmer needs to plan for the first kind and can *reduce* the second kind with a soil test. That difference will drive the "next best check" feature in Lesson 8.

New climate numbers here (the challenge asks for frost and season length):
- **Growing degree days (base 10 °C), May–Sep**: total heat available to a crop. Corn hybrids are chosen by this number.
- **Last spring frost / first fall frost**: the last day in spring and first day in fall when the minimum temperature reached 0 °C.
- **Frost-free days**: the gap between them; it limits which crops and cover crops fit.


In [ ]:
from fieldsense.sources.soilgrids import topsoil_summary
from fieldsense.profile import climate_summary, soil_summary, soil_moisture_summary

climate = climate_summary(weather, years=(1996, 2025),
                          rain_evidence="power_precip_corrected", temp_evidence="power_temperature")
soil = soil_summary(topsoil_summary(soil_table), evidence="soilgrids_topsoil")
moisture = soil_moisture_summary(smap, evidence="smap_l4_rootzone") if smap is not None else None

def show(block, title):
    rows = [{"": k, "low": v["low"], "mid": v["mid"], "high": v["high"], "unit": v["unit"], "evidence": v["evidence"]}
            for k, v in block.items() if isinstance(v, dict) and "mid" in v]
    print(title); display(pd.DataFrame(rows).set_index(""))

show(climate, f"Climate {climate['years'][0]}-{climate['years'][1]} (low/high = 10th/90th percentile of years)")
show(soil, "Topsoil 0-30 cm (low/high = SoilGrids 5%-95%)")
print("USDA texture class:", soil.get("texture_class", {}).get("class"))
if moisture:
    m = moisture["rootzone_by_month"]
    print("Root-zone wetness (m3/m3), median by month:", {k: v["mid"] for k, v in m.items()})

**Sanity checks (do these with your own knowledge, not mine):**
- Annual rain near 900 mm and growing degree days near 1,300–1,500 are normal for Wellington County. Way off? Something upstream is wrong.
- Last spring frost around mid-May (day ~130–140) and first fall frost around early October (day ~275–285) are typical for Elora.
- Is the texture class what Michelle and Maurizio would expect for the Elora plots? If SoilGrids says loam and they know it's a silt loam, that's exactly the kind of local knowledge the app should let them enter, and the evidence label would then become **Measured**.


## Step 9: Build one field profile per plot (the data contract)

A **data contract** is an agreement between two parts of a system about the exact format of what passes between them. Here:

- **Producer:** this notebook (and later, the backend) writes `profiles/plot_N.json`.
- **Consumer:** the website (Nikolett and Daniella) reads it.

As long as both respect the format, each side can change everything else independently. During the hackathon, the CS students can build the website from these files **before** the rotation engine is finished. That's how five people work in parallel without blocking each other.

The format has a **version** (`"schema": "0.1"`). If we change it later, we bump the version so the website knows what to expect.

`check_profile` enforces the contract's rules automatically: required sections present, every range ordered `low ≤ mid ≤ high`, and **every number has an evidence id**.

**The `gaps` list** is the other half of honesty: what we *don't* know. The challenge explicitly asks for information gaps, so we write them down now rather than inventing them on a slide later.


In [ ]:
from fieldsense.profile import build_field_profile, check_profile, save_profile

def plot_location(p):
    """Centre and area from the private survey boundaries if present, else the station point."""
    geo = os.path.join(PRIVATE_DIR, "elora_rotation_plots.geojson")
    if os.path.exists(geo):
        feat = [f for f in json.load(open(geo))["features"] if f["properties"]["plot"] == p][0]
        xy = np.array(feat["geometry"]["coordinates"][0])
        lat0 = np.radians(xy[:, 1].mean())
        x, y = np.radians(xy[:, 0]) * 6371000 * np.cos(lat0), np.radians(xy[:, 1]) * 6371000
        area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))) / 10000
        return round(xy[:-1, 1].mean(), 5), round(xy[:-1, 0].mean(), 5), round(area, 2), "survey map"
    return LAT, LON, None, "research station point (approximate)"

GAPS = [
    "Corn and soybean are not told apart from greenness; the farmer's own record is needed.",
    "Temperature has not yet been checked against a local station.",
    "Soil comes from a 250 m prediction with a wide organic-carbon range; a lab soil test would replace it.",
    "Soil moisture is one value per 9 km square; wet and dry spots within the field are not seen.",
    "Crop rules were tuned at Elora; accuracy at other sites is unknown until checked there.",
]
if smap is None:
    GAPS.append("No soil moisture in this run (Lesson 3 output not found).")

profiles = {}
for p in PLOTS:
    lat, lon, area, loc_src = plot_location(p)
    field = {"id": f"elora_plot_{p}", "name": f"Elora rotation plot {p}", "lat": lat, "lon": lon,
             "area_ha": area, "location_source": loc_src, "rotation_design": ROTATION.get(p),
             "region": "Wellington County, Ontario, Canada"}
    prof = build_field_profile(field, history[p], covers[p], climate=climate, soil=soil,
                               soil_moisture=moisture, barcode_file="demo/field_barcodes_elora.svg",
                               history_evidence="hls_crop_history", gaps=GAPS)
    problems = check_profile(prof)
    print(f"Plot {p}: {'OK' if not problems else problems}")
    save_profile(prof, os.path.join(PROFILE_DIR, f"plot_{p}.json"))
    profiles[p] = prof
print("Saved to", PROFILE_DIR, ":", sorted(os.listdir(PROFILE_DIR)))

**What you should see:** `OK` for every plot. If you see a problem message, it names the exact place in the profile (for example `profile.climate.frost_free_days: number has no evidence id`), which is the point of automatic checks: they tell you *where*, not just *that*.

**Privacy note:** profiles are saved in your **Drive**, not GitHub (`.gitignore` blocks the `profiles/` folder). If the survey map was used, they contain the plots' exact centres and areas. Before we put any profile in the public demo, we'll decide together whether to round the coordinates.


## Step 10: Look inside one profile

This is exactly what the website will receive. Read it slowly: everything from Lessons 1–5 is here, in one place, in a format any programming language can read.


In [ ]:
p0 = profiles[PLOTS[-1]]
preview = {
    "schema": p0["schema"],
    "field": p0["field"],
    "climate.annual_precip": p0["climate"]["annual_precip"],
    "soil.texture_class": p0["soil"].get("texture_class"),
    "history (last 3 years)": p0["history"][-3:],
    "cover (summary)": {k: v for k, v in p0["cover"].items() if k != "per_year"},
    "gaps (first 2)": p0["gaps"][:2],
}
print(json.dumps(preview, indent=2))

## Summary

Run this cell and paste its output back to me, so I can check your results before Lesson 6.


In [ ]:
print("=== Lesson 5 summary ===")
print("fieldsense", fs.__version__)
sh(f"git -C {REPO_DIR} log --oneline -1")
print("\nEvidence registry:")
for e in reg.items.values():
    print(f"  {e.label:9s} {e.id:24s} {e.result}")
print("\nSatellite crop history (since 2018):")
print(all_hist[all_hist.year >= 2018].pivot(index="year", columns="plot", values="satellite_says").map(short).to_string())
print("\nLiving cover since 2018, by rotation:")
print(summary.to_string())
print("\nClimate:", {k: climate[k]["mid"] for k in ("annual_precip", "growing_degree_days_may_sep", "frost_free_days")})
print("Soil texture:", soil.get("texture_class", {}).get("class"))
print("Profiles OK:", all(not check_profile(p) for p in profiles.values()))
print("Demo files:", sorted(os.listdir(DEMO_DIR)))

## What you learned

**ML operations**
- A **package** turns scattered notebook code into something installable, shareable and versioned.
- **Version control** (git commits) makes every result reproducible: you can name the exact code that produced it.
- **Tests** catch code bugs in seconds; **ground truth** catches wrong assumptions. You need both.
- A **data contract** (the profile schema, with a version number) lets people build different parts at the same time.

**Product design**
- Show a long history **in one glance** (the barcode), not in a table.
- Put **trust on every number** (evidence labels), and write down the **gaps**.
- Separate "the weather varies" (plan for it) from "we're not sure" (reduce it with a check).

**What's next, Lesson 6: climate futures.** We'll use 30 years of corrected weather to generate **100 possible next seasons** for the field, and add the frost, heat and extreme-rain indicators the challenge asks for. That's what turns "here's your field's past" into "here are your odds next year".
